# Checkout conversion A/B test

## 1. Business problem

An e-commerce company wants to improve the conversion rate of its checkout process. The product team has designed a new checkout experience intended to reduce friction and make it easier for customers to complete their purchases. An A/B test will be conducted to determine whether the redesigned checkout page improves conversion with the existing checkout page.

### Experiment groups

- Control group(A): Existing checkout experience
- Treatment group(B): Redesigned checkout experience

### Primary business question

Does the redesigned checkout page increase the percentage of users who complete a purchase?


## 2. Primary KPI

The primary metric for the experiment is checkout conversion rate.

The experiment will evaluate whether the treatment group achieves a higher conversion rate than the control group.

## 3. Hypotheses

### Null hypotheses (H₀)
The conversion rate of the redesigned checkout is equal to the conversion rate of the existing checkout.
H₀: pB = pA

### Alternative hypotheses (H₁)
The redesigned chekcout produces a higher conversion rate than the existing chekout.
H₁: pB > pA

## 4. MDE
The company's current checkout conversion rate is approximately 10%. For this experiment, an increase from 10% to 11% will be considered commercially meaningful.

Baseline conversion rate: 10%

Minimum target conversion rate: 11%

Minimum detectable effect: +1 percentage point

## 5. Experiment parameters

Baseline conversion rate: 10%

Minimum detectable effect: +1 percentage point

Significance level: 0.05

Statistical power: 0.80

## 6. Sample size calculation 
Before running the experiment the required sample size is calculated based on the expected baseline conversion rate, minimum detectable effect, significance level, and statistical power.

Experiment assumptions:
- Baseline conversion rate: 10%
- Target conversion rate: 11%
- Minimum detectable effect: +1 percentage point
- Significance level: 5%
- Statistical power: 80%
- Traffic allocation: 50/50

In [3]:
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize

In [4]:
control_rate = 0.10
treatment_rate = 0.11

alpha = 0.05
power = 0.80

In [5]:
effect_size = proportion_effectsize(
    treatment_rate,
    control_rate
)

effect_size

np.float64(0.03262940076737697)

In [7]:
power_analysis = NormalIndPower()

sample_size = power_analysis.solve_power(
    effect_size=effect_size,
    power=power,
    alpha=alpha,
    ratio=1,
    alternative="larger"
)

sample_size

11613.949805879798

In [8]:
import numpy as np

In [10]:
sample_size_per_group = int(np.ceil(sample_size))

sample_size_per_group

11614

In [11]:
total_sample_size = sample_size_per_group * 2\

print("Required users per group: ", sample_size_per_group)
print("required total sample size: ", total_sample_size)

Required users per group:  11614
required total sample size:  23228


## 7. Sample size decision
The power analysis estimated that approximately 11,614 users are required in each experiment group. To provide a small buffer for potential data_quality issues, the experiment will target:

- Control group: 12,000 users
- Treatment group: 12,000 users
- Total planned sample: 24,000 users

## 8. Import and initial inspection
This project uses a synthetic checkout experiment dataset which contains deliberate data_quality issues. Before calculating conversion rates, I will inspect the data and check whether its records are suitable for analysis. 

In [12]:
from pathlib import Path
import pandas as pd

In [14]:
checkout_df = pd.read_csv("../1_data/01_raw/checkout_ab_test_raw.csv")


In [15]:
checkout_df.shape

(24055, 9)

In [16]:
checkout_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 24055 entries, 0 to 24054
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   user_id        24049 non-null  str    
 1   variant        24055 non-null  str    
 2   assigned_at    24037 non-null  str    
 3   device         24020 non-null  str    
 4   channel        24025 non-null  str    
 5   customer_type  24055 non-null  str    
 6   converted      24031 non-null  str    
 7   revenue_usd    24055 non-null  float64
 8   refunded       24047 non-null  float64
dtypes: float64(2), str(7)
memory usage: 2.9 MB


In [17]:
checkout_df.isna().sum()

user_id           6
variant           0
assigned_at      18
device           35
channel          30
customer_type     0
converted        24
revenue_usd       0
refunded          8
dtype: int64

In [18]:
checkout_df.duplicated().sum()

np.int64(48)

In [19]:
checkout_df["variant"].value_counts(dropna=False)

variant
control       11970
treatment     11965
Control          24
B                21
A                21
treatment        20
Treatment        18
control          16
Name: count, dtype: int64

In [21]:
checkout_df["converted"].value_counts(dropna=False)

converted
0      21335
1       2606
No        81
NaN       24
Yes        9
Name: count, dtype: int64

## 9. Data validation and cleaning

In [23]:
#removing repeated rows and standardizing labels
clean = checkout_df.drop_duplicates().copy()

clean["user_id"] = (
    clean["user_id"]
    .astype("string")
    .str.strip()
    .replace("", pd.NA)
)

clean["variant"] = (
    clean["variant"]
    .astype("string")
    .str.strip()
    .str.lower()
    .map({
        "a": "control",
        "control": "control",
        "b": "treatment",
        "treatment": "treatment"
    })
)

clean["converted"] =(
    clean["converted"]
    .astype("string")
    .str.strip()
    .str.lower()
    .map({
        "0": 0,
        "no": 0,
        "1": 1,
        "yes": 1
    })
)

print("Rows after removing exact duplicates:", len(clean))
print("Missing user IDs:", clean["user_id"].isna().sum())
print("Unrecognized group:", clean["variant"].isna().sum())
print("Missing purchase outcomes:", clean["converted"].isna().sum())

Rows after removing exact duplicates: 24007
Missing user IDs: 6
Unrecognized group: 0
Missing purchase outcomes: 24


In [25]:
#removing records that connot be assiged reliably
missing_identity = clean["user_id"].isna() | clean["variant"].isna()
print("Rows without a usable ID or group:", missing_identity.sum())

clean= clean.loc[~missing_identity].copy()

group_counts = clean.groupby("user_id")["variant"].nunique()
conflicting_ids = group_counts[group_counts > 1].index

print("User appearing in both groups:", len(conflicting_ids))

clean = clean.loc[
    ~clean["user_id"].isin(conflicting_ids)
].copy()

Rows without a usable ID or group: 6
User appearing in both groups: 7


In [27]:
#checking missing purchase outcomes
print(
    clean.groupby("variant")["converted"]
    .apply(lambda values: values.isna().sum())
)

variant
control      14
treatment    10
Name: converted, dtype: int64


In [28]:
print("Rows with missing purchase outcomes:", clean["converted"].isna().sum())

clean = clean.loc[clean["converted"].notna()].copy()
clean["converted"] = clean["converted"].astype(int)

print("Users available for conversion analysis:", len(clean))
print(clean["variant"].value_counts())
print("One row per user:", clean["user_id"].is_unique)

Rows with missing purchase outcomes: 24
Users available for conversion analysis: 23963
variant
treatment    11983
control      11980
Name: count, dtype: int64
One row per user: True


In [30]:
#flagging issues with other metrics
clean["assigned_at"] = pd.to_datetime(
    clean["assigned_at"],
    format="mixed",
    errors="coerce"
)

clean["revenue_usd"] = pd.to_numeric(
    clean["revenue_usd"],
    errors="coerce"
)

clean["refunded"] = pd.to_numeric(
    clean["refunded"],
    errors="coerce"
)

clean["revenue_issue"] = (
    clean["revenue_usd"].isna()
    | ((clean["converted"] == 0) & (clean["revenue_usd"] != 0))
    | ((clean["converted"] == 1) & (clean["revenue_usd"] <= 0))
)

clean["refund_issue"] = (
    (clean["converted"] == 0) & (clean["refunded"] == 1)
)

print("Mising timestamps:", clean["assigned_at"].isna().sum())
print("Revenue inconsistancies:", clean["revenue_issue"].sum())
print("Refund inconsistancies:", clean["refund_issue"].sum())
print("Missing refund values:", clean["refunded"].isna().sum())

Mising timestamps: 18
Revenue inconsistancies: 18
Refund inconsistancies: 8
Missing refund values: 8
